**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/yang-2026-flexible"], check=True)
    os.chdir("/content/encore/entries/yang-2026-flexible")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Yang (2026) on DANDI:000020

**Paper.** Yang Z. (2026), "Flexible Cellular Phenotypes Retain a Shared Type-Level Organization across
Response Regimes", *eNeuro* 13(11), doi:10.1523/eneuro.0275-26.2026. The claims file,
`claims/yang-2026-flexible.yaml`, was written from the version of record (PMC13737997), with the
legends of the Extended Data, and the paper's code at commit 61a5a41
(https://github.com/zy-neuro/protocol-response-geometry).

**Dataset.** DANDI:000020, "Patch-seq recordings from mouse visual cortex" (Allen Institute,
Gouwens et al. 2020), version **0.210913.1639**, doi:10.48324/dandi.000020/0.210913.1639. This is
the version the paper cites (the README of its code, "Data") and the only published version of the
dandiset; the draft has the same 4,435 files and size. Every file is streamed into memory from the
archive and never written to disk.

**Outside inputs** (claims file, "Outside data"). The paper's join of the NWB files to the Allen
Patch-seq metadata (t-type, subclass, Cre line), `results/m0/m0_molecular_join.csv`, and the
retained feature lists, `results/phenotype/phenotype_summary.json`, both from the paper's code at
commit 61a5a41. The NeMO count matrix (release 20200611) is needed by the transcriptomic claims;
its server did not answer during this attempt (section "Transcriptomic claims").

| Claim | Tested here |
|---|---|
| join-link (added by the author) | tested |
| cohort-label-join | tested |
| table1-phenotype-spaces | tested |
| pc1-long-pulse-loadings | tested |
| fig1b-distance-correlation | tested |
| fig1-fixed-five-pcs-ordering | tested |
| fig1c-knn-jaccard | tested |
| knn-scale-sweep | tested |
| split-half-reliability | tested |
| fig2c-eta2-ttype-over-cre | tested |
| table2-alignment-ttype | tested |
| ttype-beyond-cre | tested |
| fig2d-knn-type-enrichment | tested |
| fig2ab-ttype-structure-within-cre | tested |
| fig3a-type-map-concordance | tested |
| shared-map-model-call | tested |
| subclass-map-concordance | tested |
| equal-n-centroid-robustness | tested |
| within-regime-type-map-reliability | tested |
| fig3b-pair-tags | tested |
| pair-tag-threshold-sensitivity | tested |
| type-level-exceeds-cell-level | tested |
| mobility-distribution | tested |
| fig4ab-rna-tracks-shared-map, fig4-alternative-panels, fig4c-joint-excitability-modules, fig4c-kv3-strongest, fig4c-no-module-tracks-rank-range, pvalb-marker-control | blocked: the NeMO count matrix could not be fetched |
| wgcna-mobility-within-type | blocked: the NeMO count matrix could not be fetched; WGCNA (R) is not in the entry's environment |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

In [ ]:
import json, resource, time, warnings
from collections import Counter

import numpy as np
import pandas as pd
import requests
from scipy.stats import kruskal

import patchseq as ps
import geometry as g

warnings.filterwarnings("ignore")
KEY = ps.PAPER_KEY
SEED_OFFSETS = [0, 1000, 2000, 3000, 4000]
CODE = "https://raw.githubusercontent.com/zy-neuro/protocol-response-geometry/61a5a413daea3b3562fad6e40207320da5665fed"


def rng_for(claim, offset=0, k=0):
    # the claim's generator (protocol): crc32 of "<paper>/<claim>", plus 1 per extra generator, plus the seed offset
    return np.random.default_rng(ps.seed_of(claim) + offset + k)


def peak_gb():
    return (resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)


T0 = time.time()
meta = requests.get(f"{ps.API}/dandisets/{ps.DANDISET}/versions/{ps.VERSION}/", timeout=120).json()
versions = requests.get(f"{ps.API}/dandisets/{ps.DANDISET}/versions/", timeout=120).json()["results"]
import ipfx
from numpy._core._multiarray_umath import __cpu_features__
print("IPFX", ipfx.__version__, "| numpy", np.__version__, "| CPU sort features:", {k: __cpu_features__.get(k) for k in ("AVX2", "AVX512F", "AVX512_SKX")})
print("dandiset:", meta["name"], "| version", ps.VERSION, "| doi", meta.get("doi"))
for v in versions:
    print(f"  version {v['version']}: {v['asset_count']} assets, {v['size'] / 1e9:.1f} GB, modified {v['modified'][:10]}")
assets = ps.list_assets()
print("assets listed:", len(assets), "| total", round(sum(a["size"] for a in assets) / 1e9, 1), "GB")
print("file suffixes:", Counter(a["path"].rsplit("_", 1)[-1] for a in assets))
print("subjects (folders):", len({a["path"].split("/")[0] for a in assets}))
print("description (start):", meta.get("description", "")[:400])

## Extraction

One pass over the 4,435 files, in eight processes: each file is fetched into memory, opened as an IPFX
data set, and reduced to the cell-level features of the three regimes (long squares, short squares,
ramps), the split-half long-square features of claim split-half-reliability, and the values of the
data check. The method and where each part comes from are in the docstring of `patchseq.py`.

In [ ]:
t = time.time()
# split halves under the criterion's seed 0 (the code's), the four seeds 1000 to 4000, and the claim's seed (protocol)
SPLIT_SEEDS = [0, 1000, 2000, 3000, 4000, ps.seed_of("split-half-reliability")]
recs = ps.extract_all(assets, split_seed=SPLIT_SEEDS, workers=8, progress=500)
print(f"extraction: {len(recs)} files in {(time.time() - t) / 60:.1f} min; peak memory (kB) self, largest child:", peak_gb())
print("extract_ok:", Counter(r["extract_ok"] for r in recs))
print("errors:", Counter(r["error"].split(":")[0] for r in recs if r["error"]))
for r in recs:
    if r["error"]:
        print("  ", r["path"], r["error"][:150])

The extraction reads each file's sweep table and series with h5py in place of pynwb, to save time
(`patchseq._fast_reader`). The next cell checks that this gives the same features, clock values and
split halves as IPFX's own pynwb reader on eight files spread over the dandiset.

In [ ]:
check_idx = np.linspace(0, len(assets) - 1, 8).astype(int)
same_all = True
for i in check_idx:
    a = assets[i]
    f1 = ps.extract_cell(a, ps.seed_of("split-half-reliability"), fast=True)
    f2 = ps.extract_cell(a, ps.seed_of("split-half-reliability"), fast=False)
    keys_same = f1["features"].keys() == f2["features"].keys()
    vals_same = keys_same and all(np.isclose(f1["features"][k], f2["features"][k], equal_nan=True) for k in f1["features"])
    split_same = (f1.get("split") or {}).get("a") == (f2.get("split") or {}).get("a") and (f1.get("split") or {}).get("b") == (f2.get("split") or {}).get("b")
    same_all &= bool(vals_same and split_same and f1["clock"] == f2["clock"])
    print(a["path"], "| features", len(f1["features"]), "same:", vals_same, "| split same:", split_same, "| clock same:", f1["clock"] == f2["clock"], "| errors:", f1["error"] or "-", f2["error"] or "-")
print("fast reader agrees with IPFX's reader on all checked files:", same_all)

## Data check

In [ ]:
ok = [r for r in recs if r["extract_ok"] == 1]
print("files:", len(recs), "| opened and extracted:", len(ok))
# sessions and subjects, from the files against the paths
path_ses = [r["path"].split("_ses-")[1].split("_")[0] for r in ok]
print("session id in file equals the path's:", sum(r["session_id"] == s for r, s in zip(ok, path_ses)), "of", len(ok))
print("subject id in file equals the path's:", sum(("sub-" + r["subject_id"]) == r["path"].split("/")[0] for r in ok), "of", len(ok))
print("distinct sessions:", len(set(path_ses)), "| distinct subjects:", len({r["subject_id"] for r in ok}))
print("cells per subject:", Counter(Counter(r["subject_id"] for r in ok).values()).most_common(8))
print("sex:", Counter(r["sex"] for r in ok))
ages = pd.Series([float(r["age"].strip("P").rstrip("D").split(".")[0]) if r["age"].startswith("P") else np.nan for r in ok])
print("age (days): median", ages.median(), "range", ages.min(), "to", ages.max(), "| missing", int(ages.isna().sum()))
cre_nwb = Counter(r["genotype"].split(";")[0].split("/")[0] for r in ok)
print("Cre driver in the files' genotype (first allele):", len(cre_nwb), "lines;", cre_nwb.most_common(12))
print("MIES files:", Counter(r.get("mies", True) for r in ok))
# sweeps per regime (current clamp, with experiment and stim epochs)
ns = pd.DataFrame([r["n_sweeps"] for r in ok])
print("cells with at least one usable sweep: long squares", int((ns.long_squares > 0).sum()),
      "| short squares", int((ns.short_squares > 0).sum()), "| ramps", int((ns.ramps > 0).sum()))
print("usable sweeps per cell, median (range): long", ns.long_squares.median(), (ns.long_squares.min(), ns.long_squares.max()),
      "| short", ns.short_squares.median(), (ns.short_squares.min(), ns.short_squares.max()),
      "| ramp", ns.ramps.median(), (ns.ramps.min(), ns.ramps.max()))
print("regime extraction failed (IPFX state), of", len(ok), ":", {k: sum(r["failed"][k] for r in ok) for k in ok[0]["failed"]})

In [ ]:
# Units and ranges of the features the phenotype spaces use
summary = json.loads(requests.get(f"{CODE}/results/phenotype/phenotype_summary.json", timeout=120).text)
paper_cols = {p: list(summary[p]["features"]) for p in g.REGIMES}
rows = []
for p in g.REGIMES:
    stems_p, x = g.feature_matrix(recs, paper_cols[p])
    for j, c in enumerate(paper_cols[p]):
        v = x[:, j][np.isfinite(x[:, j])]
        rows.append({"regime": p, "feature": c.split(".", 2)[2], "n": v.size, "min": v.min() if v.size else np.nan,
                     "median": np.median(v) if v.size else np.nan, "max": v.max() if v.size else np.nan})
with pd.option_context("display.max_rows", 200, "display.width", 200):
    print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
# Clock: the voltage response of each cell's most hyperpolarizing long square starts with the stimulus.
clk = pd.DataFrame([dict(r["clock"], stem=r["file_stem"]) for r in ok if r.get("clock")])
print("cells with a hyperpolarizing long square:", len(clk), "of", len(ok))
print("stimulus onset (s) from the stimulus trace:", Counter(clk.start.round(2)).most_common(4), "| duration (s):", Counter(clk.dur.round(2)).most_common(3))
print("amplitude (pA): median", clk.amp.median(), "range", clk.amp.min(), clk.amp.max())
print("voltage change relative to baseline (mV), median [5th, 95th percentile]:")
for col, what in [("pre", "10 ms just before onset"), ("post", "15 to 25 ms after onset"), ("dv", "last 100 ms of the step")]:
    print(f"  {what}: {clk[col].median():.2f} [{clk[col].quantile(0.05):.2f}, {clk[col].quantile(0.95):.2f}]")
print("cells whose response is negative 15 to 25 ms after onset:", int((clk.post < -1).sum()), "| more than 1 mV off before onset:", int((clk.pre.abs() > 1).sum()))
# Recordings deposited twice: identical voltage traces (SHA-1) of that sweep, of all long squares, or identical feature vectors
for col in ("fingerprint", "long_square_hash"):
    groups = clk.groupby(col).stem.apply(list)
    print(f"files sharing an identical {col}:", [v for v in groups if len(v) > 1])
fv = {}
for r in ok:
    if r["features"]:
        fv.setdefault(tuple(sorted((k, round(v, 9)) for k, v in r["features"].items())), []).append(r["file_stem"])
dup_feat = [v for v in fv.values() if len(v) > 1]
print("files with identical IPFX feature vectors:", dup_feat)
dups = sorted({tuple(v) for v in clk.groupby("long_square_hash").stem.apply(list) if len(v) > 1} | {tuple(v) for v in dup_feat})
feat_of = {r["file_stem"]: r["features"] for r in ok}
for grp in dups:
    fa, fb = (feat_of[s] for s in grp[:2])
    shared = sorted(set(fa) & set(fb))
    n_same = sum(1 for k in shared if (fa[k] == fb[k]) or (np.isnan(fa[k]) and np.isnan(fb[k])))
    print(f"{grp[0]} and {grp[1]}: IPFX features {len(fa)} and {len(fb)}, in both {len(shared)}, equal in both {n_same}")


## Labels: the paper's join (outside input)

### join-link (added by the author)

> The NWB files of the dandiset are the cells of the paper's metadata join (results/m0/m0_molecular_join.csv of the paper's code at commit 61a5a41); matched by file stem, they have the same session, subject and genotype, and the same membership of the three regimes.

In [ ]:
join_df = pd.read_csv(f"{CODE}/results/m0/m0_molecular_join.csv", dtype=str, keep_default_na=False)
join = {r["file_stem"]: r for r in join_df.to_dict("records")}
stems_all = [r["file_stem"] for r in recs]
print("join rows:", len(join_df), "| distinct stems:", join_df.file_stem.nunique(), "| columns:", list(join_df.columns))
matched = [s for s in stems_all if s in join]
print("dandiset files with a join row:", len(matched), "of", len(stems_all), f"({len(matched) / len(stems_all):.1%})")
print("join rows without a dandiset file:", len(set(join) - set(stems_all)))
by = {r["file_stem"]: r for r in ok}
agree = Counter()
for s in matched:
    if s not in by:
        continue
    r, j = by[s], join[s]
    agree["session"] += r["session_id"] == j["session_id"]
    agree["subject"] += r["subject_id"] == j["subject_id"]
    agree["genotype"] += r["genotype"] == j["full_genotype"]
    agree["cre_in_genotype"] += j["cre_line"].split("/")[0].split(";")[0] in r["genotype"]
    agree["n"] += 1
print("matched and extracted:", agree["n"], "| same session id:", agree["session"], "| same subject id:", agree["subject"],
      "| same full genotype:", agree["genotype"], "| join Cre line found in the file's genotype:", agree["cre_in_genotype"])
print("missing in the dandiset (no file):", sorted(set(join) - set(stems_all))[:5])
print("files with no join row:", [s for s in stems_all if s not in join])
for grp in dups:
    print("identical recordings:", [(s, join.get(s, {}).get("cre_line"), join.get(s, {}).get("t_type"), join.get(s, {}).get("structure"),
                                     join.get(s, {}).get("transcriptomics_sample_id")) for s in grp])

### cohort-label-join

> Of the cells, 4,430 have a row in the joined metadata table and 4,266 a non-empty t-type; the cells with a t-type number 4,183 in long-pulse, 3,988 in ramp and 4,266 in brief-pulse.

The per-regime counts need the phenotype spaces of claim table1-phenotype-spaces, built next; they are printed in that claim's section.

In [ ]:
n_join = len(matched)
n_ttype = sum(1 for s in matched if join[s]["t_type"].strip())
print("cells with a join row:", n_join, "| with a non-empty t-type:", n_ttype, "| join rows with a t-type:", int((join_df.t_type.str.strip() != "").sum()))

## Phenotype spaces

### table1-phenotype-spaces

> Built independently per regime, the phenotype spaces hold 4,344 long-pulse cells with 42 features and k80 = 11 components, 4,139 ramp cells with 20 features and k80 = 3, and 4,422 brief-pulse cells with 21 features and k80 = 5; the same-cell overlaps are 4,054 (long-pulse and ramp), 4,336 (long-pulse and brief-pulse), 4,134 (ramp and brief-pulse) and 4,049 (all three).

First the paper's rule for retained features (at most 30% missing, nonzero variance) applied to the
features extracted here, under both denominators (reading 2a, cells with a value in the regime; 2b,
all extract_ok cells), against the paper's lists; then the spaces on the paper's lists, as the
criterion says, and the same on the rule's lists as a variant.

In [ ]:
rule_lists = {}
for p in g.REGIMES:
    for den in ("present", "all"):
        keep, miss = g.rule_features(recs, p, denominator=den)
        rule_lists[(p, den)] = keep
        extra = sorted(set(keep) - set(paper_cols[p]))
        lost = sorted(set(paper_cols[p]) - set(keep))
        print(f"{p}, missingness over {den} cells: rule keeps {len(keep)} of {len(miss)} candidate features; paper's list {len(paper_cols[p])};"
              f" same list: {set(keep) == set(paper_cols[p])}")
        if extra:
            print("    kept here, not in the paper's list:", [c.split('.', 2)[2] for c in extra])
        if lost:
            print("    in the paper's list, not kept here:", [(c.split('.', 2)[2], round(miss.get(c, np.nan), 3)) for c in lost])
spaces = {p: g.phenotype_space(recs, paper_cols[p]) for p in g.REGIMES}
spaces_rule = {p: g.phenotype_space(recs, rule_lists[(p, "present")]) for p in g.REGIMES}
sets = {p: set(spaces[p]["stems"]) for p in g.REGIMES}
print()
for label, sp in (("paper's lists", spaces), ("rule's lists (variant)", spaces_rule)):
    st = {p: set(sp[p]["stems"]) for p in g.REGIMES}
    print(label)
    for p in g.REGIMES:
        print(f"  {p}: cells {len(sp[p]['stems'])}, features {len(sp[p]['columns'])}, k80 {sp[p]['k80']}"
              f" (cumulative variance at k80 {np.cumsum(sp[p]['var'])[sp[p]['k80'] - 1]:.3f}, at k80 - 1 {np.cumsum(sp[p]['var'])[sp[p]['k80'] - 2]:.3f})")
    print("  shared: long-ramp", len(st["long_pulse"] & st["ramp"]), "| long-brief", len(st["long_pulse"] & st["brief_pulse"]),
          "| ramp-brief", len(st["ramp"] & st["brief_pulse"]), "| all three", len(st["long_pulse"] & st["ramp"] & st["brief_pulse"]))
# against the paper's own regime membership (join columns in_long_pulse, in_ramp, in_brief_pulse)
for p in g.REGIMES:
    theirs = {s for s, r in join.items() if r[f"in_{p}"] == "1"}
    print(f"{p}: paper's join lists {len(theirs)} cells in the regime; here {len(sets[p])}; in both {len(theirs & sets[p])};"
          f" only paper {len(theirs - sets[p])}; only here {len(sets[p] - theirs)}")
print("cells with a t-type per regime: long", sum(join.get(s, {}).get("t_type", "").strip() != "" for s in spaces["long_pulse"]["stems"]),
      "| ramp", sum(join.get(s, {}).get("t_type", "").strip() != "" for s in spaces["ramp"]["stems"]),
      "| brief", sum(join.get(s, {}).get("t_type", "").strip() != "" for s in spaces["brief_pulse"]["stems"]))

**Result: reproduced.** The paper's rule applied to the features extracted here keeps exactly the paper's lists (42, 20 and 21 features) under both denominators, and the spaces hold 4,344, 4,139 and 4,422 cells with k80 = 11, 3 and 5; the overlaps are 4,054, 4,336, 4,134 and 4,049, all equal to the paper's. The cells of each regime are the paper's own cells, as listed in its join. For cohort-label-join, the cells with a t-type number 4,183, 3,988 and 4,266 per regime.

**Result: reproduced.** All 4,430 rows of the paper's join match a file of the dandiset by file stem, and every matched file has the session id, subject id and full genotype of its row. Five files have no row: the four of subject 636828314 that IPFX cannot open and one other (sub-650125704_ses-653191386). The regime membership computed here equals the join's in_long_pulse, in_ramp and in_brief_pulse columns for every cell (above: 4,344, 4,139 and 4,422 in both, none on one side only).

**Result: reproduced.** 4,430 files have a join row and 4,266 a non-empty t-type; the per-regime counts with a t-type, printed above, are 4,183, 3,988 and 4,266. All five counts equal the paper's.

### pc1-long-pulse-loadings

> Long-pulse PC1 loads most strongly on rheobase or threshold current and on input resistance.

In [ ]:
lp = spaces["long_pulse"]
load = pd.DataFrame({"feature": [c.split(".", 2)[2] for c in lp["columns"]], "loading": lp["loadings"][0]})
load["abs"] = load.loading.abs()
load = load.sort_values("abs", ascending=False).reset_index(drop=True)
current = {"threshold_i", "rheobase_i", "rheobase_sweep.stim_amp", "hero_sweep.stim_amp"}
resist = {"ri", "input_resistance"}
load["kind"] = np.where(load.feature.isin(current), "current", np.where(load.feature.isin(resist), "input resistance", "other"))
print(load.head(12).round(3).to_string())
kinds = [k for k in load.kind if k != "other"]
first_two_kinds = []
for k in load.kind:
    if k not in first_two_kinds:
        first_two_kinds.append(k)
print("PC1 explains", round(lp["var"][0], 3), "of the variance")
print("reading (a), the two highest-ranked kinds of feature:", first_two_kinds[:2])
print("reading (b), kinds among the top five:", sorted(set(load.kind[:5])))

**Result: reproduced.** The four rheobase and threshold current features have the largest absolute PC1 loadings (0.303 each), followed by the two input resistance features (0.263), so the claim holds under both readings.

## Cell-level conservation

### fig1b-distance-correlation

> Pairwise cell-cell phenotypic distances are positively correlated across all three regime pairs, above shuffle and well below identity, strongest for long-pulse and ramp (Spearman rho 0.434), then ramp and brief-pulse (0.384), then long-pulse and brief-pulse (0.336).

In [ ]:
claim = "fig1b-distance-correlation"
dc = {}
for a, b in g.PAIRS:
    stems_ab, xa, xb = g.shared(spaces[a], spaces[b])
    res = g.distance_correlation(xa, xb, rng_for(claim), n_shuffle=100, unique=True)
    var = g.distance_correlation(xa, xb, rng_for(claim, k=1), n_shuffle=100, unique=False)
    seeds = [g.distance_correlation(xa, xb, rng_for(claim, off), n_shuffle=0, unique=True)["rho"] for off in SEED_OFFSETS]
    dc[(a, b)] = res
    print(f"{a} vs {b}: n = {res['n']}, rho = {res['rho']:.4f}, p = {res['p']:.4f} (null mean {res['null_mean']:.4f}, max {res['null_max']:.4f});"
          f" code's pair draw rho = {var['rho']:.4f} ({var['n_unique_pairs']} unique of {var['n_pairs']});"
          f" five seeds {min(seeds):.4f} to {max(seeds):.4f}")
order = sorted(dc, key=lambda k: -dc[k]["rho"])
print("order:", [f"{a}-{b}" for a, b in order])

**Result: reproduced.** On unique pairs, rho is 0.441 (long-pulse and ramp), 0.385 (ramp and brief-pulse) and 0.336 (long-pulse and brief-pulse), each within 10% of the paper's 0.434, 0.384 and 0.336, each with p = 0.0099 against 100 shuffles (null maxima at most 0.025), in the paper's order. The code's pair draw gives 0.438, 0.381 and 0.334, and the five seeds keep each value within tolerance (0.435 to 0.441, 0.379 to 0.385, 0.333 to 0.343).

### fig1-fixed-five-pcs-ordering

> The ordering of cross-regime distance correlations is unchanged when a fixed five principal components are kept in each regime instead of k80.

In [ ]:
claim = "fig1-fixed-five-pcs-ordering"
dc5 = {}
for a, b in g.PAIRS:
    stems_ab, xa, xb = g.shared(spaces[a], spaces[b], 5, 5)
    seeds = [g.distance_correlation(xa, xb, rng_for(claim, off), n_shuffle=0, unique=True)["rho"] for off in SEED_OFFSETS]
    dc5[(a, b)] = seeds[0]
    print(f"{a} vs {b}, five PCs: rho = {seeds[0]:.4f} (five seeds {min(seeds):.4f} to {max(seeds):.4f})")
print("order with five PCs:", [f"{a}-{b}" for a, b in sorted(dc5, key=lambda k: -dc5[k])])

**Result: reproduced.** With five components in every regime, rho is 0.434, 0.383 and 0.339 for long-ramp, ramp-brief and long-brief, the same order as with k80, under all five seeds.

### fig1c-knn-jaccard

> Local neighborhoods are much less stable; the mean Jaccard overlap of each cell's 15 nearest neighbors across regimes is 0.023 (long-pulse and ramp), 0.022 (long-pulse and brief-pulse) and 0.019 (ramp and brief-pulse), only modestly above index-shuffle chance of about 0.002.

In [ ]:
claim = "fig1c-knn-jaccard"
kj = {}
for a, b in g.PAIRS:
    stems_ab, xa, xb = g.shared(spaces[a], spaces[b])
    res = g.knn_jaccard(xa, xb, 15, rng_for(claim))
    chances = [res["chance"]] + [g.knn_jaccard(xa, xb, 15, rng_for(claim, off))["chance"] for off in SEED_OFFSETS[1:]]
    kj[(a, b)] = res
    print(f"{a} vs {b}: n = {res['n']}, mean Jaccard = {res['mean']:.4f} (median {res['median']:.4f}), chance = {res['chance']:.5f}"
          f" (five seeds {min(chances):.5f} to {max(chances):.5f}), adjusted {res['adjusted']:.4f}")

**Result: reproduced.** The mean 15-NN Jaccard is 0.0229, 0.0224 and 0.0185 for long-ramp, long-brief and ramp-brief (paper 0.023, 0.022, 0.019), and the index-shuffle chance is 0.0019 to 0.0020 in every pair under all five seeds (paper about 0.002).

### knn-scale-sweep

> Across neighborhood sizes k = 15 to 600, chance-adjusted overlap rises with k but stays low even at k = 600 (0.134, 0.101 and 0.108 for long-pulse and ramp, long-pulse and brief-pulse, ramp and brief-pulse).

In [ ]:
claim = "knn-scale-sweep"
grid = [15, 30, 60, 120, 240, 480, 600]
scale = []
for a, b in g.PAIRS:
    stems_ab, xa, xb = g.shared(spaces[a], spaces[b])
    rng = rng_for(claim)
    for k in grid:
        res = g.knn_jaccard(xa, xb, k, rng)
        scale.append({"pair": f"{a}-{b}", "k": k, "mean": res["mean"], "chance": res["chance"], "adjusted": res["adjusted"]})
    adj600 = [g.knn_jaccard(xa, xb, 600, rng_for(claim, off))["adjusted"] for off in SEED_OFFSETS[1:]]
    print(f"{a}-{b}: adjusted at k = 600 under the other four seeds: {np.round(adj600, 4)}")
scale = pd.DataFrame(scale)
print(scale.pivot(index="k", columns="pair", values="adjusted").round(4).to_string())
for pair, sub in scale.groupby("pair"):
    adj = sub.sort_values("k").adjusted.values
    print(pair, "| rises at every step:", bool(np.all(np.diff(adj) > 0)), "| k=600 above k=15:", bool(adj[-1] > adj[0]))

**Result: reproduced.** The chance-adjusted overlap at k = 600 is 0.134, 0.100 and 0.108 (paper 0.134, 0.101, 0.108), within 0.0003 under the other four seeds, and it rises at every step of the k grid for each pair.

### split-half-reliability

> Within long-pulse, neighborhoods built from two disjoint halves of each cell's sweeps overlap more (mean 15-NN Jaccard 0.109, chance about 0.0026, adjusted 0.107, 3,598 cells) than cross-regime neighborhoods in the same pool (0.025, 0.024 and 0.020), but far from completely.

The halves were extracted in the pass above (the code's `extract_one_split`). Its random split
draws its permutation from a generator made with seed 0 for each cell, so the seed decides which
cells enter the pool: the criterion gives "random (seed 0)", and that is the main result here. The
same is shown under the seeds 1000 to 4000 and under the claim's seed (protocol), with which the
first version of this notebook ran. Each half is put in the paper's long-pulse columns, missing
values filled with the full-cell feature mean, z-scored with the full-cell mean and SD and
projected on the full-cell loadings (k80 components). The pool is compared, cell by cell, with the
paper's own list of the split-half cells (`results/q1_reliability/split_half_jaccard_cells.csv` of
its code), which is used for that comparison only.


In [ ]:
claim = "split-half-reliability"
sp_recs = [r for r in ok if r.get("split")]
print("cells with at least four usable long squares:", sum(r["split"]["n_usable"] >= ps.MIN_USABLE for r in sp_recs))
lp = spaces["long_pulse"]
cols = lp["columns"]
paper_cells = set(pd.read_csv(f"{CODE}/results/q1_reliability/split_half_jaccard_cells.csv", usecols=["file_stem"])["file_stem"])
print("paper's split-half cells:", len(paper_cells))


def project(x):
    xf = np.where(np.isfinite(x), x, lp["mu"])
    return ((xf - lp["mu"]) / lp["sd"]) @ lp["loadings"].T[:, : lp["k80"]]


def split_half(seed, show=False):
    # a cell stopped before any split was tried (too few sweeps, no subthreshold amplitude) has no per-seed result
    res_by = [(r, r["split"]["by_seed"].get(seed, r["split"])) for r in sp_recs]
    halves = []
    for r, s in res_by:
        if not s["ok"]:
            continue
        a, b = (s["a"], s["b"]) if s["rule"] == "random" else (r["split"]["a"], r["split"]["b"])
        halves.append((r["file_stem"], a, b))
    xa = np.array([[a.get(c, np.nan) for c in cols] for _, a, _ in halves], dtype=float)
    xb = np.array([[b.get(c, np.nan) for c in cols] for _, _, b in halves], dtype=float)
    keep = np.isfinite(xa).any(axis=1) & np.isfinite(xb).any(axis=1)
    stems_h = [s for (s, _, _), k in zip(halves, keep) if k]
    res = g.knn_jaccard(project(xa[keep]), project(xb[keep]), 15, rng_for(claim))
    out = {"seed": seed, "n": len(stems_h), "within": res["mean"], "chance": res["chance"], "adjusted": res["adjusted"],
           "rules": Counter(s["rule"] for _, s in res_by if s["ok"]),
           "reasons": Counter(s["reason"].split(":")[0][:40] for _, s in res_by if not s["ok"]),
           "in_paper": len(set(stems_h) & paper_cells), "only_here": len(set(stems_h) - paper_cells),
           "only_paper": len(paper_cells - set(stems_h))}
    for a, b in g.PAIRS:
        pool = [s for s in stems_h if s in sets[a] and s in sets[b]]
        pos_a = {s: i for i, s in enumerate(spaces[a]["stems"])}
        pos_b = {s: i for i, s in enumerate(spaces[b]["stems"])}
        xa_ = spaces[a]["scores"][[pos_a[s] for s in pool], : spaces[a]["k80"]]
        xb_ = spaces[b]["scores"][[pos_b[s] for s in pool], : spaces[b]["k80"]]
        r2 = g.knn_jaccard(xa_, xb_, 15, rng_for(claim, k=1), n_shuffle=1)
        out[f"n {a}-{b}"] = len(pool)
        out[f"J {a}-{b}"] = r2["mean"]
    if show:
        chances = [g.knn_jaccard(project(xa[keep]), project(xb[keep]), 15, rng_for(claim, off))["chance"] for off in SEED_OFFSETS[1:]]
        out["chance range over five shuffle seeds"] = (min([res["chance"]] + chances), max([res["chance"]] + chances))
    return out


split_runs = [split_half(s, show=(s == 0)) for s in SPLIT_SEEDS]
main = split_runs[0]
print(f"seed 0 (criterion): both halves extracted and projected {main['n']}; rules {dict(main['rules'])}")
print("  reasons for no split:", main["reasons"].most_common(6))
print(f"  within long-pulse half A against half B: mean 15-NN Jaccard {main['within']:.4f}, chance {main['chance']:.6f}"
      f" (five shuffle seeds {main['chance range over five shuffle seeds'][0]:.6f} to {main['chance range over five shuffle seeds'][1]:.6f}),"
      f" adjusted {main['adjusted']:.4f}")
for a, b in g.PAIRS:
    print(f"  same pool {a}-{b}: n = {main[f'n {a}-{b}']}, mean 15-NN Jaccard {main[f'J {a}-{b}']:.5f}")
print("  within exceeds every same-pool cross-regime value:", all(main["within"] > main[f"J {a}-{b}"] for a, b in g.PAIRS))
print(f"  cells in the paper's list too: {main['in_paper']}; here only: {main['only_here']}; paper only: {main['only_paper']}")
tab = pd.DataFrame([{k: (dict(v) if isinstance(v, Counter) else v) for k, v in r.items() if k not in ("reasons", "chance range over five shuffle seeds")}
                    for r in split_runs])
tab["random"] = [r["rules"].get("random", 0) for r in split_runs]
tab["seed"] = ["0 (criterion)", "1000", "2000", "3000", "4000", f"{SPLIT_SEEDS[-1]} (claim's seed)"]
pd.set_option("display.width", 250)
print(tab.drop(columns=["rules"]).round(5).to_string(index=False))


**Result: reproduced.** Under the criterion's seed 0 for the random split (the code's), 3,598 cells have both halves, the same 3,598 cells as the paper's own list (0 here only, 0 there only), split odd/even 3,112, sweep-number interleaved 128 and random 358, the paper's counts exactly. The within-regime mean 15-NN Jaccard is 0.1090 against chance 0.002581, adjusted 0.1064 (paper 0.109, 0.0026, 0.107), and it exceeds each same-pool cross-regime value, 0.02537, 0.02411 and 0.02011 (paper 0.025, 0.024, 0.020), on pools of 3,276, 3,509 and 3,351 cells (the paper's). The seed decides which cells enter the pool: under the seeds 1000 to 4000 the pool has 3,741, 4,100, 3,582 and 3,853 cells (random splits 501, 860, 342 and 613), and under the claim's seed 3,256 (random 16), the values of the first version of this notebook, with the ramp and brief-pulse value 0.02219 just outside its tolerance and the long-pulse and ramp pool 2,946. The within-regime mean exceeds the three same-pool values under every seed shown (0.1071 to 0.1107 against at most 0.0258).

## Transcriptomic type anchors the phenotype

### fig2c-eta2-ttype-over-cre

> T-type accounts for more multivariate phenotypic variance than Cre line in every regime (eta squared 0.327 against 0.209 long-pulse, 0.393 against 0.267 ramp, 0.149 against 0.110 brief-pulse).

Readings for Cre: (a) the code's set, cells with both a t-type and a Cre line; (b) all cells with a
Cre line. The join gives every row a Cre line. A variant takes the Cre line from the files' own
genotype (the dandiset), for the cells of reading (b).

In [ ]:
cre_file = {r["file_stem"]: r["genotype"].split(";")[0].split("/")[0] for r in ok}
eta = {}
for p in g.REGIMES:
    x_t, lab_t, idx_t = g.labelled(spaces[p], join, "t_type")
    cre_on_t = np.array([join[spaces[p]["stems"][i]]["cre_line"].strip() for i in idx_t], dtype=object)
    has_c = cre_on_t != ""
    e_t = g.eta2(x_t, lab_t)
    e_ca = g.eta2(x_t[has_c], cre_on_t[has_c])
    x_c, lab_c, idx_c = g.labelled(spaces[p], join, "cre_line")
    e_cb = g.eta2(x_c, lab_c)
    lab_f = np.array([cre_file[spaces[p]["stems"][i]] for i in idx_c], dtype=object)
    e_cf = g.eta2(x_c, lab_f)
    eta[p] = (e_t, e_ca, e_cb)
    print(f"{p}: t-type eta2 {e_t:.4f} (n {len(lab_t)}, {len(set(lab_t))} types) | Cre (a) {e_ca:.4f} (n {int(has_c.sum())}, {len(set(cre_on_t[has_c]))} lines)"
          f" | Cre (b) {e_cb:.4f} (n {len(lab_c)}) | Cre from the files' genotype {e_cf:.4f} ({len(set(lab_f))} lines) | t-type > Cre: {e_t > e_ca}")

**Result: reproduced.** Under reading (a), the code's cell set, eta squared of t-type against Cre is 0.327 against 0.209 (long-pulse), 0.393 against 0.267 (ramp) and 0.149 against 0.110 (brief-pulse), the paper's values to three decimals, and t-type exceeds Cre in every regime. Reading (b) and the Cre line from the files' own genotypes also give t-type above Cre in every regime.

### table2-alignment-ttype

> Pairwise alignment of binary t-type mismatch with phenotypic distance is small in all three regimes (A_t 0.021 long-pulse, 0.025 ramp, 0.021 brief-pulse) but exceeds shuffled labels in each.

In [ ]:
claim = "table2-alignment-ttype"
for p in g.REGIMES:
    x_t, lab_t, _ = g.labelled(spaces[p], join, "t_type")
    runs = [g.alignment(x_t, lab_t, rng_for(claim, off)) for off in SEED_OFFSETS]
    res = runs[0]
    avg = g.alignment(x_t, lab_t, rng_for(claim, k=1), n_shuffle=0, average_ranks=True)["A"]
    print(f"{p}: n {res['n']}, A_t {res['A']:.4f}, p {res['p']:.4f} (null mean {res['null_mean']:.4f}, max {res['null_max']:.4f});"
          f" five seeds A_t {[round(r['A'], 4) for r in runs]}, p {[round(r['p'], 4) for r in runs]};"
          f" A_t minus null mean {[round(r['A'] - r['null_mean'], 4) for r in runs]}; average-rank Spearman (variant) {avg:.4f}")
# variant: the draws of the code's q2_anchor.py under its seed 0, cells in the order of its score files
order_same = all(pd.read_csv(f"{CODE}/results/phenotype/{p}_pca_scores.csv", usecols=["file_stem"])["file_stem"].tolist() == spaces[p]["stems"]
                 for p in g.REGIMES)
print("cells here in the order of the code's score files, in all three regimes:", order_same)
code_seq = g.code_q2_sequence(spaces, join, seed=0)
for p in g.REGIMES:
    r = code_seq[p]
    print(f"{p} (code's draws, seed 0): n {r['n_t']}, A_t {r['A_t']['A']:.4f}, p {r['A_t']['p']:.4f}, null mean {r['A_t']['null_mean']:.4f},"
          f" A_t minus null mean {r['A_t']['A'] - r['A_t']['null_mean']:.4f}; residual A_t (n {r['n_c']}) {r['A_res']['A']:.4f},"
          f" p {r['A_res']['p']:.4f}, null mean {r['A_res']['null_mean']:.4f}")
# check of the method, not of the data: the same draws on the paper's own published PCA scores
# (results/phenotype/*_pca_scores.csv), against the paper's printed A_t and null mean (results/q2/q2_anchor.csv)
paper_spaces = {}
for p in g.REGIMES:
    df = pd.read_csv(f"{CODE}/results/phenotype/{p}_pca_scores.csv")
    paper_spaces[p] = {"stems": df["file_stem"].tolist(), "scores": df[[c for c in df.columns if c.startswith("PC")]].to_numpy(),
                       "k80": spaces[p]["k80"]}
paper_seq = g.code_q2_sequence(paper_spaces, join, seed=0)
q2 = pd.read_csv(f"{CODE}/results/q2/q2_anchor.csv").set_index("protocol")
for p in g.REGIMES:
    r = paper_seq[p]["A_t"]
    print(f"{p}: paper's scores, code's draws: A_t {r['A']:.4f}, null mean {r['null_mean']:.4f}, A_t minus null mean {r['A'] - r['null_mean']:.4f};"
          f" paper's printed A_t {q2.loc[p, 'A_t_type']:.4f}, null mean {q2.loc[p, 'A_t_type_null']:.4f},"
          f" A_t minus null mean {q2.loc[p, 'A_t_type'] - q2.loc[p, 'A_t_type_null']:.4f};"
          f" this attempt's scores, code's draws: A_t minus null mean {code_seq[p]['A_t']['A'] - code_seq[p]['A_t']['null_mean']:.4f}")
# variant: the same draws with a stable sort of the ranks (tied values ranked in their order of input), to show
# how much the order the sort gives tied ranks moves A_t and its null mean
stable_paper = g.code_q2_sequence(paper_spaces, join, seed=0, kind="stable")
stable_here = g.code_q2_sequence(spaces, join, seed=0, kind="stable")
for p in g.REGIMES:
    for what, d, base in (("paper's scores", stable_paper, paper_seq), ("this attempt's scores", stable_here, code_seq)):
        r, b = d[p]["A_t"], base[p]["A_t"]
        print(f"{p}, {what}, code's draws, stable sort: A_t {r['A']:.4f} (default sort {b['A']:.4f}), null mean {r['null_mean']:.4f}"
              f" (default {b['null_mean']:.4f}), A_t minus null mean {r['A'] - r['null_mean']:.4f} (default {b['A'] - b['null_mean']:.4f})")


**Result: partially reproduced.** A_t is 0.034, 0.028 and 0.014 for long-pulse, ramp and brief-pulse (paper 0.021, 0.025, 0.021), each above all 100 label shuffles (p = 0.0099), so the alignment exceeds shuffled labels in each regime but none of the three values is within its tolerance. The value moves with the sampled pairs: over the five seeds it ranges 0.021 to 0.034 (long-pulse), 0.024 to 0.034 (ramp) and 0.014 to 0.025 (brief-pulse), with p = 0.0099 under every seed, while A_t minus its null mean is stable, 0.0280 to 0.0284, 0.0265 to 0.0287 and 0.0162 to 0.0182. With the code's own draws under its seed 0, A_t is 0.0292, 0.0294 and 0.0177 here; on the paper's own published scores the same draws give 0.0370, 0.0316 and 0.0106, not the printed 0.0214, 0.0250 and 0.0213, but A_t minus the null mean there, 0.0264, 0.0261 and 0.0161, is within 0.0005 of the paper's printed A_t minus its printed null mean, 0.0264, 0.0266 and 0.0166. So the printed A_t differs from what the code's draws give on the paper's scores by an offset that its null mean shares. The order the sort gives tied ranks is one thing that moves A_t and its null mean together: with a stable sort the code's draws give A_t 0.0350, 0.0301 and 0.0109 on the paper's scores and 0.0313, 0.0308 and 0.0180 here, changes of at most 0.0021, while A_t minus the null mean changes by at most 0.0003. These shifts are much smaller than the gap to the printed A_t (0.016 for long-pulse), and what produces that gap is not shown here. This attempt's A_t minus null mean (0.0272, 0.0252, 0.0188 under the code's draws) is within 0.0022 of the paper's. The tie-corrected Spearman (variant) is 0.069 to 0.096, larger, as the claims file expected.

### ttype-beyond-cre

> After removing each Cre line's mean phenotype, t-type alignment stays above the shuffled-label null in all three regimes, though weaker (A_t 0.011, 0.010 and 0.004 for long-pulse, ramp and brief-pulse), and residual t-type eta squared remains detectable.

In [ ]:
claim = "ttype-beyond-cre"
for p in g.REGIMES:
    x_t, lab_t, idx_t = g.labelled(spaces[p], join, "t_type")
    cre = np.array([join[spaces[p]["stems"][i]]["cre_line"].strip() for i in idx_t], dtype=object)
    has = cre != ""
    xr = g.residualize(x_t[has], cre[has])
    runs = [g.alignment(xr, lab_t[has], rng_for(claim, off)) for off in SEED_OFFSETS]
    res = runs[0]
    e_res = g.eta2(xr, lab_t[has])
    r2 = rng_for(claim, k=1)
    null = [g.eta2(xr, lab_t[has][r2.permutation(has.sum())]) for _ in range(100)]
    print(f"{p}: n {int(has.sum())}, residual A_t {res['A']:.4f}, p {res['p']:.4f} (null mean {res['null_mean']:.4f});"
          f" five seeds A_t {[round(r['A'], 4) for r in runs]}, p {[round(r['p'], 4) for r in runs]};"
          f" A_t minus null mean {[round(r['A'] - r['null_mean'], 4) for r in runs]};"
          f" residual t-type eta2 {e_res:.4f} against label-shuffle null mean {np.mean(null):.4f} (max {np.max(null):.4f})")
# the code's draws under its seed 0 (computed in table2-alignment-ttype), on this attempt's scores and on the paper's
for p in g.REGIMES:
    r, q = code_seq[p]["A_res"], paper_seq[p]["A_res"]
    print(f"{p}: code's draws, seed 0: residual A_t {r['A']:.4f} (p {r['p']:.4f}, null mean {r['null_mean']:.4f}, minus null {r['A'] - r['null_mean']:.4f});"
          f" on the paper's scores {q['A']:.4f} (null mean {q['null_mean']:.4f}, minus null {q['A'] - q['null_mean']:.4f}); paper's printed {q2.loc[p, 'A_t_type_after_cre_residual']:.4f}")


**Result: partially reproduced.** After removing each Cre line's mean, A_t is 0.008, 0.006 and 0.010 (paper 0.011, 0.010, 0.004), each with p = 0.0099, so the residual alignment is above the label null in every regime, but none of the three values is within its tolerance; over the five seeds the values range -0.002 to 0.010, and A_t minus its null mean 0.0042 to 0.0087, 0.0039 to 0.0067 and 0.0037 to 0.0049. With the code's draws under seed 0 the residual A_t is 0.0112, 0.0052 and 0.0038 (p = 0.0099 each), long-pulse and brief-pulse within tolerance and ramp not; on the paper's own scores the same draws give 0.0145, 0.0048 and 0.0070 against the printed 0.0114, 0.0102 and 0.0041, so, as for A_t, the code's draws on the paper's scores do not give the printed values. Residual t-type eta squared, 0.110, 0.120 and 0.040, is above the largest of 100 label-shuffle values in each regime (0.018, 0.021, 0.031), so residual structure is detectable under both readings.

### fig2d-knn-type-enrichment

> Cells are enriched for phenotypic neighbors of their own t-type, most for long-pulse (6.59-fold), then ramp (3.44-fold) and brief-pulse (2.64-fold).

In [ ]:
enr = {}
for p in g.REGIMES:
    x_t, lab_t, _ = g.labelled(spaces[p], join, "t_type")
    res = g.knn_enrichment(x_t, lab_t)
    enr[p] = res["ratio_of_means"]
    print(f"{p}: n {res['n']}, same-type fraction {res['same']:.4f}, chance {res['chance']:.4f}, enrichment (ratio of means) {res['ratio_of_means']:.3f},"
          f" mean of per-cell ratios (variant) {res['mean_of_ratios']:.3f}")
print("order long > ramp > brief:", enr["long_pulse"] > enr["ramp"] > enr["brief_pulse"])

**Result: reproduced.** The same-t-type enrichment among the 15 nearest neighbours is 6.608, 3.445 and 2.643 (paper 6.59, 3.44, 2.64), in the paper's order; the mean of per-cell ratios (variant) gives 6.751, 3.565 and 2.867.

### fig2ab-ttype-structure-within-cre

> T-types occupy distinct but overlapping parts of phenotype space, and finer t-type structure remains within single Cre-driver populations; within Sst-IRES-Cre the six most frequent t-types occupy distinct positions on long-pulse PC1.

In [ ]:
claim = "fig2ab-ttype-structure-within-cre"
p = "long_pulse"
x_t, lab_t, idx_t = g.labelled(spaces[p], join, "t_type")
cre = np.array([join[spaces[p]["stems"][i]]["cre_line"].strip() for i in idx_t], dtype=object)
rng = rng_for(claim)
rows = []
for c, n in Counter(cre.tolist()).most_common():
    sel = cre == c
    if n < 80 or len(set(lab_t[sel])) < 2:
        continue
    res = g.alignment(x_t[sel], lab_t[sel], rng)
    e = g.eta2(x_t[sel], lab_t[sel])
    null = [g.eta2(x_t[sel], lab_t[sel][rng.permutation(n)]) for _ in range(100)]
    rows.append({"cre": c, "n": n, "t_types": len(set(lab_t[sel])), "A_t": res["A"], "p": res["p"], "eta2": e, "eta2_null_mean": np.mean(null)})
within = pd.DataFrame(rows)
print(within.round(4).to_string(index=False))
sst = within[within.cre.str.startswith("Sst-IRES-Cre")]
print("Sst-IRES-Cre rows:", sst.cre.tolist())
sel = cre == (sst.cre.iloc[0] if len(sst) else "")
if sel.any():
    top = [t for t, _ in Counter(lab_t[sel].tolist()).most_common(6)]
    pc1 = x_t[sel, 0]
    groups = [pc1[lab_t[sel] == t] for t in top]
    for t, gr in zip(top, groups):
        print(f"  {t}: n {gr.size}, median PC1 {np.median(gr):.3f}")
    print("  Kruskal-Wallis across the six (descriptive):", kruskal(*groups))

**Result: reproduced.** Two Cre lines of the paper's join with at least 80 cells have the Sst-IRES-Cre driver, one with the Ai14 reporter and one with Ai167 (the files' genotypes count them as one driver, cell 7). For Sst-IRES-Cre with Ai14 (993 cells, 32 t-types), within-Cre A_t is 0.031 with p = 0.0099 and t-type eta squared is 0.225 against a shuffle mean of 0.031; for Sst-IRES-Cre with Ai167 (81 cells), 0.044 with p = 0.020 and 0.488 against 0.262. The six most frequent t-types of the first have long-pulse PC1 medians from 0.23 to 1.84 (Kruskal-Wallis p = 6e-19, descriptive).

## Type-level relationships

### fig3a-type-map-concordance

> Among the 57 t-types with at least 10 cells in every regime (1,596 type pairs), type-type centroid distance maps are concordant across all three regime pairs, strongest for long-pulse and ramp (rho 0.658), then ramp and brief-pulse (0.608), then long-pulse and brief-pulse (0.455).

In [ ]:
claim = "fig3a-type-map-concordance"
lab = {p: g.labelled(spaces[p], join, "t_type") for p in g.REGIMES}
counts = {p: Counter(lab[p][1].tolist()) for p in g.REGIMES}
types = sorted(t for t in counts["long_pulse"] if all(counts[p][t] >= 10 for p in g.REGIMES))
codes = {p: g.type_codes(lab[p][1], types) for p in g.REGIMES}
S = {p: g.centroid_distances(lab[p][0], codes[p], len(types))[0] for p in g.REGIMES}
print("t-types with at least 10 cells in every regime:", len(types), "| type pairs:", len(types) * (len(types) - 1) // 2)
conc = {}
nulls = {}
for a, b in g.PAIRS:
    rho = g.spearman(g.upper(S[a]), g.upper(S[b]))
    per_seed = []
    for off in SEED_OFFSETS:
        rng = rng_for(claim, off)
        lab_b = lab[b][1]
        null = np.empty(1000)
        for t in range(1000):
            sh = lab_b[rng.permutation(lab_b.size)]
            null[t] = g.spearman(g.upper(S[a]), g.upper(g.centroid_distances(lab[b][0], g.type_codes(sh, types), len(types))[0]))
        per_seed.append(null)
    conc[(a, b)] = rho
    nulls[(a, b)] = per_seed[0]
    ps_ = [g.emp_p(nl, rho) for nl in per_seed]
    print(f"{a} vs {b}: rho {rho:.4f}, p {ps_[0]:.6f} (null mean {per_seed[0].mean():.4f}, max {per_seed[0].max():.4f}); p under five seeds {np.round(ps_, 6)}")
print("order:", [f"{a}-{b}" for a, b in sorted(conc, key=lambda k: -conc[k])])

**Result: reproduced.** 57 t-types have at least 10 cells in every regime (1,596 pairs). The concordance is 0.659 (long-ramp), 0.609 (ramp-brief) and 0.455 (long-brief), within 0.001 of the paper's 0.658, 0.608 and 0.455, each above all 1,000 shuffles (p = 0.000999 under all five seeds), in the paper's order.

### shared-map-model-call

> The three type-map concordances meet the prespecified shared-map criterion (minimum rho at least 0.40 and range across the three below 0.25).

In [ ]:
vals = np.array(list(conc.values()))
print(f"minimum {vals.min():.4f}, maximum {vals.max():.4f}, range {vals.max() - vals.min():.4f}; shared-map rule holds: {vals.min() >= 0.40 and vals.max() - vals.min() < 0.25}")

**Result: reproduced.** The minimum concordance is 0.455 (at least 0.40) and the range 0.204 (below 0.25).

### subclass-map-concordance

> At the subclass level (six subclasses), type relationships are even more conserved between long-pulse and ramp (rho 0.936) and moderately aligned for the comparisons with brief-pulse (0.579 and 0.621).

In [ ]:
sub = {p: g.labelled(spaces[p], join, "subclass") for p in g.REGIMES}
sub_types = {p: sorted(t for t, n in Counter(sub[p][1].tolist()).items() if n >= 10) for p in g.REGIMES}
print("subclasses with at least 10 cells:", {p: sub_types[p] for p in g.REGIMES})
for a, b in g.PAIRS:
    sh = sorted(set(sub_types[a]) & set(sub_types[b]))
    Sa = g.centroid_distances(sub[a][0], g.type_codes(sub[a][1], sh), len(sh))[0]
    Sb = g.centroid_distances(sub[b][0], g.type_codes(sub[b][1], sh), len(sh))[0]
    print(f"{a} vs {b}: {len(sh)} shared subclasses, rho {g.spearman(g.upper(Sa), g.upper(Sb)):.4f} (average ranks {g.spearman_avg(g.upper(Sa), g.upper(Sb)):.4f})")

**Result: reproduced.** Six subclasses have at least 10 cells in every regime; the subclass map concordance is 0.936 (long-ramp), 0.579 (long-brief) and 0.621 (ramp-brief), the paper's values under reading (a).

### equal-n-centroid-robustness

> Rebuilt from 10 cells per type per regime (200 resamples), type-map concordance drops for all three pairs (long-pulse and ramp mean 0.554, ramp and brief-pulse 0.455, long-pulse and brief-pulse 0.329) but stays well above shuffled-label nulls.

In [ ]:
claim = "equal-n-centroid-robustness"
by_type = {p: {t: np.where(codes[p] == i)[0] for i, t in enumerate(types)} for p in g.REGIMES}


def equal_n(rng, n_boot=200, n_draw=10):
    out = {pair: [] for pair in g.PAIRS}
    for _ in range(n_boot):
        Sb = {}
        for p in g.REGIMES:
            take = np.concatenate([rng.choice(by_type[p][t], size=n_draw, replace=False) for t in types])
            cc = np.repeat(np.arange(len(types)), n_draw)
            Sb[p] = g.centroid_distances(lab[p][0][take], cc, len(types))[0]
        for a, b in g.PAIRS:
            out[(a, b)].append(g.spearman(g.upper(Sb[a]), g.upper(Sb[b])))
    return {k: np.array(v) for k, v in out.items()}


eq = equal_n(rng_for(claim))
eq_seeds = [eq] + [equal_n(rng_for(claim, off)) for off in SEED_OFFSETS[1:]]
for a, b in g.PAIRS:
    v = eq[(a, b)]
    nl = nulls[(a, b)]
    means = [e[(a, b)].mean() for e in eq_seeds]
    print(f"{a} vs {b}: full {conc[(a, b)]:.4f}; equal-N mean {v.mean():.4f}, 95% interval {np.quantile(v, 0.025):.4f} to {np.quantile(v, 0.975):.4f};"
          f" five seeds mean {min(means):.4f} to {max(means):.4f}; below full: {v.mean() < conc[(a, b)]};"
          f" null max {nl.max():.4f}, null 97.5th {np.quantile(nl, 0.975):.4f}; (a) 2.5th > null max: {np.quantile(v, 0.025) > nl.max()}; (b) mean > null 97.5th: {v.mean() > np.quantile(nl, 0.975)}")

**Result: reproduced.** With 10 cells per type, the mean concordance over 200 resamples is 0.553 (long-ramp), 0.457 (ramp-brief) and 0.323 (long-brief), within the paper's spread of 0.554, 0.455 and 0.329, each below its full-sample value, and the 2.5th percentiles (0.469, 0.359, 0.203) exceed the largest of the 1,000 shuffles (0.167, 0.147, 0.186), so both readings of "well above" hold. The five seeds move the means by at most 0.008.

### within-regime-type-map-reliability

> Within-regime split-half concordance of the type-type maps (long-pulse 0.861, ramp 0.851, brief-pulse 0.799) is well above the mean full-sample cross-regime concordance (about 0.57).

In [ ]:
claim = "within-regime-type-map-reliability"


def split_half_maps(rng, n_boot=200):
    out = {p: [] for p in g.REGIMES}
    for _ in range(n_boot):
        for p in g.REGIMES:
            ta, tb, ca, cb = [], [], [], []
            for i, t in enumerate(types):
                perm = rng.permutation(by_type[p][t])
                h = perm.size // 2
                ta.append(perm[:h]); tb.append(perm[h:2 * h]); ca.append(np.full(h, i)); cb.append(np.full(h, i))
            Sa = g.centroid_distances(lab[p][0][np.concatenate(ta)], np.concatenate(ca), len(types))[0]
            Sb = g.centroid_distances(lab[p][0][np.concatenate(tb)], np.concatenate(cb), len(types))[0]
            out[p].append(g.spearman(g.upper(Sa), g.upper(Sb)))
    return {p: np.mean(v) for p, v in out.items()}


rel_seeds = [split_half_maps(rng_for(claim, off)) for off in SEED_OFFSETS]
ref = np.mean(list(conc.values()))
for p in g.REGIMES:
    vals_p = [r[p] for r in rel_seeds]
    print(f"{p}: split-half S concordance {vals_p[0]:.4f} (five seeds {min(vals_p):.4f} to {max(vals_p):.4f}); exceeds the reference: {vals_p[0] > ref}")
print(f"reference, mean of the three full-sample rho: {ref:.4f}")

**Result: reproduced.** The split-half concordance of the type maps is 0.862, 0.852 and 0.799 (paper 0.861, 0.851, 0.799), against a mean full-sample cross-regime concordance of 0.574 (paper about 0.57), and the five seeds keep each within 0.005.

### fig3b-pair-tags

> Of the 1,596 type pairs, 639 (40.0%) are stable, 801 (50.2%) intermediate and 156 (9.8%) regime-selective.

In [ ]:
tags_, cv_, range_, mean_ = g.pair_tags(S)
tc = Counter(tags_.tolist())
print({k: (v, f"{v / len(tags_):.1%}") for k, v in tc.items()}, "of", len(tags_))

**Result: reproduced.** 644 pairs are stable, 796 intermediate and 156 selective (paper 639, 801, 156).

### pair-tag-threshold-sensitivity

> The proportions of stable, intermediate and selective pairs are stable under modest variation of the classification thresholds.

In [ ]:
base = {k: v / len(tags_) for k, v in tc.items()}
rows = []
for sel_cut in [0.45, 0.50, 0.55, 0.60, 0.65]:
    t2 = g.pair_tags(S, 0.25, sel_cut)[0]
    rows.append({"varied": "selective range", "cutoff": sel_cut, **{k: np.mean(t2 == k) for k in ("stable", "intermediate", "selective")}})
for cv_cut in [0.15, 0.20, 0.25, 0.30, 0.35]:
    t2 = g.pair_tags(S, cv_cut, 0.55)[0]
    rows.append({"varied": "stable CV", "cutoff": cv_cut, **{k: np.mean(t2 == k) for k in ("stable", "intermediate", "selective")}})
sens = pd.DataFrame(rows)
for k in ("stable", "intermediate", "selective"):
    sens[f"change_{k}"] = sens[k] - base.get(k, 0)
print(sens.round(3).to_string(index=False))
print("selective below stable and intermediate at every cutoff:", bool(((sens.selective < sens.stable) & (sens.selective < sens.intermediate)).all()))

**Result: reproduced.** At every cutoff of both grids the selective fraction (0.038 to 0.235) is below both the stable (0.231 to 0.569) and the intermediate (0.335 to 0.671) fractions. The selective fraction moves from 0.235 to 0.038 over the range cutoffs, and the stable fraction from 0.231 to 0.569 over the CV cutoffs.

### type-level-exceeds-cell-level

> Type-level conservation across regimes is much greater than the preservation of cell-level relationships and local neighborhoods.

In [ ]:
for a, b in g.PAIRS:
    print(f"{a} vs {b}: type-map rho {conc[(a, b)]:.4f} | cell-cell distance rho {dc[(a, b)]['rho']:.4f} | 15-NN Jaccard {kj[(a, b)]['mean']:.4f}"
          f" | type exceeds cell: {conc[(a, b)] > dc[(a, b)]['rho']}")

**Result: reproduced.** For each regime pair the type-map concordance (0.659, 0.455, 0.609) exceeds the cell-cell distance correlation (0.441, 0.336, 0.385), and the 15-NN Jaccard is about 0.02.

## Mobility

### mobility-distribution

> Cell-level neighborhood mobility across regimes, one minus the mean Spearman correlation of a cell's distance profile across the three regime pairs, has mean 0.589 and SD 0.110 over the 4,049 cells in all three regimes.

In [ ]:
mob_stems, mob, mob_rhos = g.mobility(spaces)
print(f"cells in all three regimes: {len(mob_stems)}; mobility mean {mob.mean():.4f}, SD {mob.std():.4f} (ddof 0), range {mob.min():.4f} to {mob.max():.4f}")
print("mean per-pair profile correlation:", {f"{a}-{b}": round(float(mob_rhos[:, j].mean()), 4) for j, (a, b) in enumerate(g.PAIRS)})

**Result: reproduced.** Over the 4,049 cells in all three regimes, mobility has mean 0.589 and SD 0.110, the paper's values.

## Transcriptomic claims

These claims need the NeMO Patch-seq count matrix (release 20200611,
`20200513_Mouse_PatchSeq_Release_count.v2.csv`). The cell below asks the NeMO data server for the
release folder.

In [ ]:
url = "https://data.nemoarchive.org/other/AIBS/AIBS_patchseq/transcriptome/scell/SMARTseq/processed/analysis/20200611/"
for attempt in range(3):
    try:
        r = requests.get(url, timeout=60)
        print("NeMO answered", r.status_code, len(r.content), "bytes")
    except Exception as exc:
        print("NeMO request failed:", type(exc).__name__, str(exc)[:200])
    time.sleep(5)
print(f"notebook ran {(time.time() - T0) / 60:.1f} min; peak memory (kB) self, largest child:", peak_gb())

In [ ]:
# The two cell counts of wgcna-mobility-within-type that need no count matrix: triple-shared cells
# with a transcriptomic sample id and a t-type, and those in t-types with at least three of them
# (the code's q3_wgcna_mobility.py, through the claim's criterion). Other values need the matrix.
rna = [s for s in mob_stems if join.get(s, {}).get("transcriptomics_sample_id", "").strip() and join[s]["t_type"].strip()]
tc3 = Counter(join[s]["t_type"].strip() for s in rna)
print(f"triple-shared cells {len(mob_stems)}; with an RNA sample id and a t-type {len(rna)} (paper 3,906);"
      f" in t-types with at least three such cells {sum(n for n in tc3.values() if n >= 3)} (paper 3,902)")


**Result: blocked.** The NeMO data server did not return the release folder (the cell above prints its answers), so the seven transcriptomic claims (fig4ab-rna-tracks-shared-map, fig4-alternative-panels, fig4c-joint-excitability-modules, fig4c-kv3-strongest, fig4c-no-module-tracks-rank-range, pvalb-marker-control and wgcna-mobility-within-type) are not tested. The two cell counts of wgcna-mobility-within-type that need no count matrix are printed above: 3,906 triple-shared cells with an RNA sample id and a t-type and 3,902 in t-types of at least three, the paper's counts (whether every one of these samples is in the count matrix could not be checked); the claim stays blocked, as its other values need the matrix. The WGCNA claim would also need R and WGCNA 1.74, which the entry's Python environment does not hold.

## Summary

| Claim | Outcome |
|---|---|
| join-link | reproduced |
| cohort-label-join | reproduced |
| table1-phenotype-spaces | reproduced |
| pc1-long-pulse-loadings | reproduced |
| fig1b-distance-correlation | reproduced |
| fig1-fixed-five-pcs-ordering | reproduced |
| fig1c-knn-jaccard | reproduced |
| knn-scale-sweep | reproduced |
| split-half-reliability | reproduced (under the criterion's seed 0 for the random split; seed-sensitive) |
| fig2c-eta2-ttype-over-cre | reproduced |
| table2-alignment-ttype | partially reproduced |
| ttype-beyond-cre | partially reproduced |
| fig2d-knn-type-enrichment | reproduced |
| fig2ab-ttype-structure-within-cre | reproduced |
| fig3a-type-map-concordance | reproduced |
| shared-map-model-call | reproduced |
| subclass-map-concordance | reproduced |
| equal-n-centroid-robustness | reproduced |
| within-regime-type-map-reliability | reproduced |
| fig3b-pair-tags | reproduced |
| pair-tag-threshold-sensitivity | reproduced |
| type-level-exceeds-cell-level | reproduced |
| mobility-distribution | reproduced |
| the six Figure 4 claims | blocked (NeMO count matrix not fetched) |
| wgcna-mobility-within-type | blocked (NeMO count matrix not fetched; WGCNA in R not available) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The IPFX version is the current release (2.1.2),
run through its per-regime functions; the paper's driver script is not in its code. `extract_ok`
is taken as "the file opened and the extraction ran" (an assumption; every cell count still equals
the paper's). The sweeps are read with h5py in place of pynwb, checked against IPFX's own reader on
eight files. Every random draw uses the protocol's seed for its claim in place of the code's seed
0, except the random split-half split, which the criterion gives as seed 0 and which decides the
pool of that claim; it is also shown under five other seeds. The A_t values depend on the seed and
on the order the sort gives tied ranks (a stable sort is shown as a variant), and the code's own draws under seed 0 are shown as a variant.
The run time and the extraction time are printed above (the cell before the transcriptomic claims, and the extraction cell); the extraction is almost all of it. Pairs for the distance correlation
are unique pairs (the Methods), with the code's draw as a variant.

**What was found about the dataset.** Everything the paper describes is in the files: 4,435 NWB
files, each labelled as one cell (two of them hold the same recording, below), of which 4,431 open in IPFX; the four files of subject 636828314 have two
response series for one sweep number, and IPFX cannot read them. Sessions and subjects in the
files equal those of the paths, and the files' genotypes equal the paper's join. The response of
every cell's most hyperpolarizing long square starts at the stimulus onset given by the stimulus
trace (data check), so stimulus and response share a clock. One pair of files from different mice,
with different Cre lines and t-types in the paper's join (sub-673135474_ses-680563483 and
sub-673135516_ses-680897463), holds identical IPFX feature vectors; the data check prints the
files whose long-square voltage traces are identical. The files hold no transcriptomic label,
sample id or Cre-line field of their own: the t-types come from the Allen metadata, through the
paper's join.